In [5]:
from pathlib import Path
import os
import shutil


In [6]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

In [7]:
project_dir = Path(os.getcwd()).parent.parent
install_dir = project_dir / "install"
log_dir = project_dir / "logs" / "dlio"
data_dir = Path("/p/lustre5/haridev/dlio_demo")
output_dir = project_dir / "output" / "dlio"
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /usr/WS2/haridev/dftracer-demo/install
Log Directory: /usr/WS2/haridev/dftracer-demo/logs/dlio
Data Directory: /p/lustre5/haridev/dlio_demo
Output Directory: /usr/WS2/haridev/dftracer-demo/output/dlio


In [8]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


In [9]:
import os

import importlib.util

spec = importlib.util.find_spec("dftracer")
if spec and spec.origin:
    print("dftracer module path:", spec.origin)
    dftracer_folder = os.path.dirname(spec.origin)
    print("dftracer folder:", dftracer_folder)
else:
    print("dftracer module not found.")

dftracer module path: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer/__init__.py
dftracer folder: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer


In [10]:
queue="pdebug"
tasks=2

[Documentation](https://dftracer.readthedocs.io/en/latest/api.html)

In [13]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1


export DFTRACER_BIND_SIGNALS=1

export DFTRACER_TRACE_COMPRESSION=1

echo "Activating environment"
source {project_dir}/demo/dlio/setup_env.sh {install_dir} 2> /dev/null

echo "Running DLIO Generation with DFTracer"
flux alloc -n {tasks} -o fastload -q {queue} {install_dir}/bin/dlio_benchmark workload=unet3d_a100 ++workload.workflow.generate_data=True ++workload.workflow.train=False hydra.run.dir={output_dir}/unet3d_a100-gen/ ++workload.output.folder={output_dir}/unet3d_a100-gen/  ++workload.dataset.num_files_train=32 ++workload.dataset.record_length_bytes=1048576 ++workload.dataset.data_folder={data_dir}/unet3d_a100/data ++workload.checkpoint.checkpoint_folder={data_dir}/unet3d_a100/checkpoint ++workload.train.batch_size=1 ++workload.train.epochs=1 > {output_dir}/log_data_gen.txt 2> {output_dir}/error_data_gen.txt || true
echo "DLIO Generation done."

Configuring DFTracer
Activating environment


Running DLIO Generation with DFTracer


In [14]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1


export DFTRACER_BIND_SIGNALS=1

export DFTRACER_TRACE_COMPRESSION=1

echo "Activating environment"
source {project_dir}/demo/dlio/setup_env.sh {install_dir} 2> /dev/null

# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

echo "Running DLIO with DFTracer"
flux alloc -n {tasks} -o fastload -q {queue} {install_dir}/bin/dlio_benchmark workload=unet3d_a100 hydra.run.dir={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.dataset.num_files_train=32 ++workload.dataset.record_length_bytes=1048576 ++workload.dataset.data_folder={data_dir}/unet3d_a100/data ++workload.checkpoint.checkpoint_folder={data_dir}/unet3d_a100/checkpoint ++workload.train.batch_size=1 ++workload.train.epochs=1 > {output_dir}/log.txt 2> {output_dir}/error.txt || true
echo "Finished running DLIO with DFTracer"

Configuring DFTracer
Activating environment


Running DLIO with DFTracer


In [18]:
import glob

pfw_files = glob.glob(str(output_dir/ "unet3d_a100" / "*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)


Found .pfw.gz files:
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0-of-1.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-a7d7eec45f522fc6-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-d13d45a6ec608eee-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-21ddb25b81a7762f-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-e177f71b091511c0-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-4cf1e279166700af-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-bbfa84573debffbb-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-11d3f4a03e2fa9a8-app.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-74487043262e7a99-app.pfw.gz


In [16]:
%%pybash
{install_dir}/bin/dftracer_split -n unet3d -f -d {output_dir}/unet3d_a100 -o {output_dir}/unet3d_a100/compact

Arguments:
  App name: unet3d
  Override: 1
  Data dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100
  Output dir: /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact
  Chunk size: 1024


14/08/2025 13:59:41 Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
14/08/2025 13:59:41 Found zq executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zq
14/08/2025 13:59:41 sqlite3 exists
14/08/2025 13:59:41  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100: 9
14/08/2025 13:59:41  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
14/08/2025 13:59:41  Removing existing indices as override is passed.
14/08/2025 13:59:41  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-21ddb25b81a7762f-app.pfw.gz
14/08/2025 13:59:41  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-a7d7eec45f522fc6-app.pfw.gz
14/08/2025 13:59:41  Created index for file /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-

rm: cannot remove '/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/counting.bak': No such file or directory


14/08/2025 13:59:48 Completed collecting size .03125095 8 of 9                                
14/08/2025 13:59:48 Finished collecting data from 9 tasks
14/08/2025 13:59:48 Scheduled chunks: 0
14/08/2025 13:59:48 Total chunks: 1
14/08/2025 13:59:48 Start processing chunks
14/08/2025 14:01:44 Chunk 1 out of 1 done with size 5.65543934 MB, path = /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact/unet3d-1.pfw                               
14/08/2025 14:01:44 All chunks processed
14/08/2025 14:01:44 re-index split files
14/08/2025 14:01:44  Number of *.pfw* files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/compact: 1
14/08/2025 14:01:44  Found zindex executable at /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/zindex_py/bin/zindex
14/08/2025 14:01:44  Removing existing indices as override is passed.
14/08/2025 14:01:44  Compressing file unet3d-1
14/08/2025 14:01:44  Created index for compressed file /usr/WS2/haridev/dftracer-demo/outpu

14/08/2025 14:01:52 Error: Original lines count 24223 does not match split lines count 24219. Please check the file. 
14/08/2025 14:01:52 Done re-index of split files


In [17]:
!gzip -dc {output_dir}/unet3d_a100/compact/*.pfw.gz | (head -n 10; echo "..."; tail -n 5)

[
{"id":1,"name":"HH","cat":"dftracer","pid":338832,"tid":338832,"ph":"M","args":{"hhash":"529e4188b6d277c8","name":"tuolumne1010","value":"529e4188b6d277c8"}}
{"id":2,"name":"thread_name","cat":"dftracer","pid":338832,"tid":338832,"ph":"M","args":{"hhash":"529e4188b6d277c8","name":"338832","value":"thread_name"}}
{"id":3,"name":"SH","cat":"dftracer","pid":338832,"tid":338832,"ph":"M","args":{"hhash":"529e4188b6d277c8","name":"/usr/WS2/haridev/dftracer-demo/install/bin/python;/usr/WS2/haridev/dftracer-demo/install/bin/dlio_benchmark;workload=unet3d_a100;hydra.run.dir=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.dataset.num_files_train=32;++workload.dataset.record_length_bytes=1048576;++workload.dataset.data_folder=/p/lustre5/haridev/dlio_demo/unet3d_a100/data;++workload.checkpoint.checkpoint_folder=/p/lus

In [12]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/unet3d_a100/compact/",
        
    ]
)
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                         ┃ Unit                  ┃                  Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                       │ seconds               │                 16.150 │
│ Total Count                                                    │ count                 │                 23,912 │
│ Total Files                                                    │ count                 │                      4 │
│ Total Nodes                                                    │ count                 │                      0 │
│ Total Processes                                                │ count                 │                      5 │
│ POSIX Count                                                    │ count                 │                 23,912 │
│ POSIX Size                                                     │ MB                    │               5485.648 │
│ POSIX Bandwidth                                                │ MB/s                  │               2850.105 │
│ POSIX Avg Transfer Size                                        │ MB                    │                  0.229 │
└────────────────────────────────────────────────────────────────┴───────────────────────┴────────────────────────┘
                                                  Layer Breakdown                                                  
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer      ┃        Time (s) ┃          Ops ┃          Ops/sec ┃        Size (MB) ┃            Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ POSIX      │           1.925 │       23,912 │        12423.638 │         5485.648 │                    2850.105 │
└────────────┴─────────────────┴──────────────┴──────────────────┴──────────────────┴─────────────────────────────┘

In [4]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/unet3d_a100/compact/",
        f"analyzer/preset=dlio",
        
    ]
)
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:112: RuntimeWarning: invalid value encountered in scalar divide
  ops=float('nan') if pd.isna(time) else float(count / time),
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:113: RuntimeWarning: invalid value encountered in scalar divide
  bandwidth=float('nan') if pd.isna(time) or pd.isna(size) else float(size / time),


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                                     ┃ Unit            ┃            Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                                   │ seconds         │           16.150 │
│ Total Count                                                                │ count           │           24,078 │
│ Total Files                                                                │ count           │                6 │
│ Total Nodes                                                                │ count           │                0 │
│ Total Processes                                                            │ count           │                5 │
│ App Count                                                                  │ count           │                1 │
│ Training Count                                                             │ count           │                1 │
│ Compute Count                                                              │ count           │                4 │
│ Fetch Data Count                                                           │ count           │                3 │
│ Data Loader Count                                                          │ count           │               32 │
│ Data Loader Fork Count                                                     │ count           │               12 │
│ Reader Count                                                               │ count           │              112 │
│ Reader POSIX (Lustre) Count                                                │ count           │           23,898 │
│ Reader POSIX (Lustre) Size                                                 │ MB              │         5485.648 │
│ Reader POSIX (Lustre) Bandwidth                                            │ MB/s            │         3175.189 │
│ Reader POSIX (Lustre) Avg Transfer Size                                    │ MB              │            0.230 │
│ Checkpoint Count                                                           │ count           │                1 │
│ Checkpoint POSIX (Lustre) Count                                            │ count           │                2 │
│ Other POSIX Count                                                          │ count           │               12 │
└────────────────────────────────────────────────────────────────────────────┴─────────────────┴──────────────────┘
                                          Layer Breakdown (w/ overlap %)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Layer                       ┃       Time (s) ┃            Ops ┃    Ops/sec ┃       Size (MB) ┃ Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ App                         │  13.307 (----) │       1 (----) │      0.075 │               - │                - │
│ Training                    │  13.243 (----) │       1 (----) │      0.076 │               - │                - │
│ Compute                     │   2.545 (----) │       4 (----) │      1.572 │               - │                - │
│ Fetch Data                  │  10.659 (  0%) │       3 ( 33%) │      0.281 │               - │                - │
│ Data Loader                 │  11.131 (  0%) │      32 (  3%) │      2.875 │               - │                - │
│ Data Loader Fork            │   0.197 (  0%) │      12 (  0%) │     61.052 │               - │                - │
│ Reader                      │  10.661 (  0%) │     112